<a href="https://colab.research.google.com/github/HarperVicic/MMA3001-Project/blob/main/Data_Input.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INPUT DATA:**

In [3]:
import pandas as pd

# Define path to the occupancy sensor dataset
file_path = '/content/5occupancySensor_MayToDec2024_9MRows.csv'

# Load the occupancy dataset into a pandas DataFrame
# Set low_memory=False to suppress mixed-type warnings and skip bad lines on parsing failure
df = pd.read_csv(file_path, on_bad_lines='skip', low_memory=False)

Let's look at the first few rows of the DataFrame to understand its structure.

In [4]:
# Render the first 5 records of the loaded occupancy dataset to inspect headers and values
display(df.head())

,id,deviceid,floorspaceid,occupancystatus,headcount,collecteddate,occupancystatuschangedate,previousoccupancystatus
0,136043,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,CurrentlyOccupied,1,2023-11-24 00:36:52+11,2023-11-24 00:36:52+11,NotOccupied
1,136048,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,RecentlyOccupied,0,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,CurrentlyOccupied
2,136050,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,NotOccupied
3,136052,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,NotOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,RecentlyOccupied
4,136053,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,CurrentlyOccupied


Now, let's get a summary of the DataFrame, including column names, non-null counts, and data types.

In [5]:
# Print comprehensive metadata summary of the occupancy dataset
# This details the index, column labels, non-null counts, and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76918 entries, 0 to 76917
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   id                         76918 non-null  int64 
 1   deviceid                   76918 non-null  object
 2   floorspaceid               76917 non-null  object
 3   occupancystatus            76916 non-null  object
 4   headcount                  76915 non-null  object
 5   collecteddate              76914 non-null  object
 6   occupancystatuschangedate  76914 non-null  object
 7   previousoccupancystatus    76913 non-null  object
dtypes: int64(1), object(7)
memory usage: 4.7+ MB


First, let's convert the `collecteddate` column to a datetime format so we can perform time-based analysis.

In [6]:
# Convert the timestamp column 'collecteddate' to standardized datetime format
# 'mixed' format is used to handle parsing variety, and invalid times are coerced to NaT
df['collecteddate'] = pd.to_datetime(df['collecteddate'], format='mixed', utc=True, errors='coerce')

# Extract just the date component for daily aggregation purposes
df['date'] = df['collecteddate'].dt.date

# Print the total number of dates that could not be successfully parsed
print(f"Number of unparseable dates: {df['collecteddate'].isna().sum()}")

# Preview original and newly parsed date formats side-by-side
display(df[['collecteddate', 'date']].head())

Number of unparseable dates: 4


,collecteddate,date
0,2023-11-23 13:36:52+00:00,2023-11-23
1,2023-11-23 13:37:32+00:00,2023-11-23
2,2023-11-23 13:37:32+00:00,2023-11-23
3,2023-11-23 13:37:52+00:00,2023-11-23
4,2023-11-23 13:37:52+00:00,2023-11-23


In [7]:
# Set path to the physical sensor and location mapping directory sheet
excel_path = '/content/Sensor ID and Locations.xlsx'

# Load the sensor registry spreadsheet containing floor designations and room identifiers
sensor_locations_df = pd.read_excel(excel_path)

# Output metadata about the location register columns and counts
print("Sensor Locations Dataset Overview:")
print(sensor_locations_df.info())

# Review the top 10 rows to inspect room metadata taxonomy
display(sensor_locations_df.head(10))

Sensor Locations Dataset Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Building        78 non-null     int64 
 1   Floor/Room      78 non-null     object
 2   Area            78 non-null     object
 3   Physical Label  78 non-null     object
 4   Sensor          78 non-null     object
 5   ID              78 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 3.8+ KB
None


,Building,Floor/Room,Area,Physical Label,Sensor,ID
0,60,1-PS01,Corridor,ES1104,Env,6012003000486
1,69,1-PS02,Breakout,ES2113,Env,6012003000882
2,60,1.40,Board Room,ES1102,Env,6012003000462
3,60,1.41,Open Office,ES1110,Env,6012003000424
4,60,1.41A,Focus Room 1,ES1101,Env,6012003000448
5,60,1.42,Startup Workshop,ES1103,Env,6012003000479
6,60,1.43,Startup,ES1105,Env,6012003000493
7,60,1.45,Meeting Room,ES1106,Env,6012003000509
8,60,1.47,Corridor,ES1107,Env,6012003000516
9,60,1.49,Workshop,ES1108,Env,6012003000530


# **Load Env. Data**

In [20]:
import json

# Set path to the environmental variable dataset
env_file_path = '/content/5EnvSensor_MayToDec2024_180kRows.csv'

# Define the necessary target columns to parse from raw data
target_columns = ['id', 'sensorid', 'jsondata', 'createdate']

# Load environmental dataset with python engine to handle complex nested string contents safely
env_df = pd.read_csv(
    env_file_path,
    usecols=target_columns,
    on_bad_lines='skip',
    engine='python'
)

# Define helper function to safely process JSON strings
def safe_json_parse(val):
    try:
        return json.loads(val) if isinstance(val, str) else {}
    except (TypeError, json.JSONDecodeError):
        return {}

# Parse nested 'jsondata' strings and normalize them into a flattened DataFrame structure
parsed_json_df = pd.json_normalize(env_df['jsondata'].apply(safe_json_parse))

# Drop raw 'jsondata' string and merge flattened attributes back with core index columns
env_df = pd.concat([env_df.drop(columns=['jsondata']), parsed_json_df], axis=1)

# Output data structure overview
print("Environmental Sensor Dataset with Expanded JSON attributes:")
print(env_df.info())

# Preview structured raw columns
display(env_df.head(5))

Environmental Sensor Dataset with Expanded JSON attributes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179447 entries, 0 to 179446
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          179447 non-null  int64 
 1   sensorid    179447 non-null  int64 
 2   createdate  179447 non-null  object
 3   0           179447 non-null  object
 4   1           179447 non-null  object
 5   2           179447 non-null  object
 6   3           179447 non-null  object
 7   4           179447 non-null  object
 8   5           179447 non-null  object
 9   6           179447 non-null  object
 10  7           179447 non-null  object
 11  8           177738 non-null  object
 12  9           177738 non-null  object
 13  10          148172 non-null  object
dtypes: int64(2), object(12)
memory usage: 19.2+ MB
None


,id,sensorid,createdate,0,1,2,3,4,5,6,7,8,9,10
0,2,6012002000241,2024-02-29 03:11:58.210358+11,"{'values': [{'time': 1702273965, 'value': 4.18...","{'values': [{'time': 1702273965, 'value': 603}...","{'values': [{'time': 1702273965, 'value': 66}]...","{'values': [{'time': 1702273965, 'value': 6}],...","{'values': [{'time': 1702273965, 'value': 1011...","{'values': [{'time': 1702273965, 'value': 23.8...","{'values': [{'time': 1702273965, 'value': 14}]...","{'values': [{'time': 1702273965, 'value': 8}],...","{'values': [{'time': 1702273965, 'value': 9}],...","{'values': [{'time': 1702273965, 'value': 9}],...","{'values': [{'time': 1702273965, 'value': 9}],..."
1,4,6012002000241,2024-02-29 17:09:08.833822+11,"{'values': [{'time': 1702273965, 'value': 4.18...","{'values': [{'time': 1702273965, 'value': 603}...","{'values': [{'time': 1702273965, 'value': 66}]...","{'values': [{'time': 1702273965, 'value': 6}],...","{'values': [{'time': 1702273965, 'value': 1011...","{'values': [{'time': 1702273965, 'value': 23.8...","{'values': [{'time': 1702273965, 'value': 14}]...","{'values': [{'time': 1702273965, 'value': 8}],...","{'values': [{'time': 1702273965, 'value': 9}],...","{'values': [{'time': 1702273965, 'value': 9}],...","{'values': [{'time': 1702273965, 'value': 9}],..."
2,5,6012002000326,2024-02-29 17:09:12.444245+11,"{'values': [{'time': 1702270857, 'value': 4.18...","{'values': [{'time': 1702270857, 'value': 525}...","{'values': [{'time': 1702270857, 'value': 68}]...","{'values': [{'time': 1702270857, 'value': 5}],...","{'values': [{'time': 1702270857, 'value': 1012...","{'values': [{'time': 1702270857, 'value': 23.8...","{'values': [{'time': 1702270857, 'value': 7}],...","{'values': [{'time': 1702270857, 'value': 6}],...","{'values': [{'time': 1702270857, 'value': 6}],...","{'values': [{'time': 1702270857, 'value': 6}],...","{'values': [{'time': 1702270857, 'value': 6}],..."
3,16,6012002000777,2024-02-29 17:10:24.443407+11,"{'values': [{'time': 1683326565, 'value': 4.15...","{'values': [{'time': 1683326565, 'value': 425}...","{'values': [{'time': 1683326565, 'value': 51}]...","{'values': [{'time': 1683326565, 'value': 1}],...","{'values': [{'time': 1683326565, 'value': 1015...","{'values': [{'time': 1683326565, 'value': 18.8...","{'values': [{'time': 1683326565, 'value': 71}]...","{'values': [{'time': 1683326565, 'value': 1}],...","{'values': [{'time': 1683326565, 'value': 3}],...","{'values': [{'time': 1683326565, 'value': 3}],...","{'values': [{'time': 1683326565, 'value': 4}],..."
4,24,6012002000869,2024-02-29 17:11:11.348578+11,"{'values': [{'time': 1709186696, 'value': 4.12...","{'values': [{'time': 1709186696, 'value': 465}...","{'values': [{'time': 1709186696, 'value': 56}]...","{'values': [{'time': 1709186696, 'value': 17}]...","{'values': [{'time': 1709186696, 'value': 1010...","{'values': [{'time': 1709186696, 'value': 24.3...","{'values': [{'time': 1709186696, 'value': 0}],...","{'values': [{'time': 1709186696, 'value': 2}],...","{'values': [{'time': 1709186696, 'value': 2}],...","{'values': [{'time': 1709186696, 'value': 2}],...","{'values': [{'time': 1709186696, 'value': 2}],..."


In [ ]:
import numpy as np

# Isolate nested dictionary columns parsed from environmental 'jsondata'
json_cols = [col for col in env_df.columns if col not in ['id', 'sensorid', 'createdate']]

extracted_data = {}

# Extract descriptive labels from nested dictionary structures dynamically
for col in json_cols:
    sample_series = env_df[col].dropna()
    if sample_series.empty:
        continue

    first_valid = sample_series.iloc[0]

    # Traverse standard metadata keys to resolve true physical measurement names
    if isinstance(first_valid, dict) and 'variable.name' in first_valid:
        var_name = first_valid['variable.name']
    elif isinstance(first_valid, dict) and 'variable' in first_valid and isinstance(first_valid['variable'], dict):
        var_name = first_valid['variable'].get('name', f"var_{col}")
    elif isinstance(first_valid, dict) and 'type' in first_valid:
        var_name = first_valid['type']
    else:
        var_name = f"var_{col}"

    # Helper to parse lists and retrieve the primary numeric values as floats
    def extract_float(x):
        if isinstance(x, dict) and 'values' in x and len(x['values']) > 0:
            try:
                val = x['values'][0].get('value')
                return float(val) if val is not None else np.nan
            except (ValueError, TypeError):
                return np.nan
        return np.nan

    # Assign float converted series to the target dictionary key
    extracted_data[var_name] = env_df[col].apply(extract_float)

# Assemble individual measurement columns into clean, uniform DataFrame
expanded_variables_df = pd.DataFrame(extracted_data)

# Concatenate key metrics with primary ID indicators
cleaned_env_df = pd.concat([
    env_df[['id', 'sensorid', 'createdate']],
    expanded_variables_df
], axis=1)

# Print final schema information
print("Environmental sensor data with actual descriptive column names as floats:")
print(cleaned_env_df.info())
display(cleaned_env_df.head(5))

# **Integrate for daily averages of headcount, Carbon dioxide, Humidity, Pressure, Temperature**

In [18]:
# Filter physical registry to locate Lab areas using keyword matching
lab_sensors = sensor_locations_df[
    sensor_locations_df['Floor/Room'].astype(str).str.contains('Lab', case=False, na=False) |
    sensor_locations_df['Area'].astype(str).str.contains('Lab', case=False, na=False)
].copy()

# Coerce registry IDs to string format for unified matching comparisons
lab_sensors['ID_str'] = lab_sensors['ID'].astype(str).str.strip()
df['deviceid_str'] = df['deviceid'].astype(str).str.strip()

# Output matching registries
print(f"Found {len(lab_sensors)} Lab rooms in the sensor locations registry.")
display(lab_sensors[['Floor/Room', 'Area', 'Physical Label', 'ID_str']])

# Identify which registered Lab IDs are actually logging events in the occupancy database
available_ids_in_data = df['deviceid_str'].unique()
matched_labs = lab_sensors[lab_sensors['ID_str'].isin(available_ids_in_data)].copy()

# Output match results showing counts per matched device
print(f"\nOut of those, {len(matched_labs)} Lab room IDs actually appear in the occupancy dataset:")
for idx, row in matched_labs.iterrows():
    device_id = row['ID_str']
    room_name = row['Floor/Room']
    sub_df = df[df['deviceid_str'] == device_id]
    valid_headcounts = pd.to_numeric(sub_df['headcount'], errors='coerce').dropna()
    print(f"- Room: {room_name} (ID: {device_id}) -> Total records: {len(sub_df)}, Valid numeric headcount values: {len(valid_headcounts)}")

Found 16 Lab rooms in the sensor locations registry.


,Floor/Room,Area,Physical Label,ID_str
13,1.65,Collab Breakout (W),ES2105,6012003000769
14,1.65,Collab Breakout (E),ES2106,6012003000776
15,1.66,Lab,ES2104,6012003000745
22,1.76,Lab,ES2115,6012003000905
23,1.77,Lab,ES2116,6012003000912
24,1.78,Lab,ES2114,6012003000899
25,69,Lab,ES2101,6012003000714
36,G.23,Collab Space (West),ES1G06,6012003000226
37,G.23,Collab Space (East),ES1G07,6012003000233
67,G.55,Lab,ES2G04,6012003000455



Out of those, 0 Lab room IDs actually appear in the occupancy dataset:


In [ ]:
"""
!pip install PyGithub
from google.colab import userdata
import os

# --- 1. Configuration ---
# Replace these with your actual GitHub details
github_username = "HarperVicic"
github_email = "hvic0003@student.monash.edu"
repo_name = "MMA3001-Project"
commit_message = "Commit from Colab workbook"

# The file you want to upload (use the path you found earlier)
file_to_upload = "Data_Input.ipynb"

# Retrieve your GitHub token from Colab Secrets
token = userdata.get('GH_TOKEN')
repo_url = f"https://{token}@github.com/{github_username}/{repo_name}.git"

# Configure Git identity
!git config --global user.email "{hvic0003@student.monash.edu}"
!git config --global user.name "{HarperVicic}"

# --- 2. Clone the Repository ---
# Clean up any existing folder with the same name to avoid conflicts
!rm -rf {repo_name}
!git clone {repo_url}

# --- 3. Prepare and Commit ---
# Navigate into the cloned repository
%cd {repo_name}

# Add, commit, and push the changes
!git add .
!git commit -m "{commit_message}"
!git push origin main

# Navigate back to the default Colab directory
%cd /content
"""

In [19]:
# Retrieve distinct device IDs present in the occupancy DataFrame
unique_occupancy_ids = df['deviceid'].dropna().unique()
print("Unique occupancy sensor IDs found:", list(unique_occupancy_ids))

# Align registry IDs to clean string values for exact lookup alignment
sensor_locations_df['ID_str'] = sensor_locations_df['ID'].astype(str).str.strip()

# Query locations DataFrame for exact matches with any active occupancy devices
matched_rooms = sensor_locations_df[sensor_locations_df['ID_str'].isin([str(x).strip() for x in unique_occupancy_ids])]

# Handle missing matches with partial identification fallbacks
if not matched_rooms.empty:
    print("\nMatching physical room details:")
    display(matched_rooms[['Floor/Room', 'Area', 'Physical Label', 'ID_str']])
else:
    print("\nNone of these occupancy sensor IDs exist in the registry spreadsheet. Checking for partial string matches...")
    # Search for partial matches using first 6 characters to handle potential ID truncations
    partial_matches = []
    for occ_id in unique_occupancy_ids:
        matches = sensor_locations_df[sensor_locations_df['ID_str'].str.contains(str(occ_id)[:6], na=False)]
        if not matches.empty:
            partial_matches.append(matches)
    if partial_matches:
        display(pd.concat(partial_matches).drop_duplicates())
    else:
        print("No partial ID matches found either.")

Unique occupancy sensor IDs found: ['56923747-b390-4f2a-8930-808429132266', '5692374+11', '56923747-b390-4f2a-8930-808429']

None of these occupancy sensor IDs exist in the registry spreadsheet. Checking for partial string matches...
No partial ID matches found either.
